# AI exposure and U.S. employment by industry

This notebook investigates whether U.S. industries with greater exposure to artificial intelligence (AI) have experienced different employment changes over time. It uses BLS industry employment data and a separate industry-level AI exposure measure. The data has not yet been added, so this notebook describes and implements the planned workflow without reporting findings.

## Data and method

Place the input CSVs described in [`data/README.md`](data/README.md) in the `data/` directory. Employment data must contain `industry`, `year`, and `employment`; AI exposure data must contain `industry` and `ai_exposure`. Industry labels must match between files. The analysis compares each industry's first and last observed employment values and relates the percentage change to its AI exposure. It is descriptive and does not establish that AI exposure caused employment changes.

The code cells use Python's standard library. If the input files are missing, the notebook reports that the analysis is not ready to run.

In [ ]:
import csv
import math
from pathlib import Path

DATA_DIR = Path("data")
EMPLOYMENT_PATH = DATA_DIR / "industry_employment.csv"
EXPOSURE_PATH = DATA_DIR / "ai_exposure.csv"
RESULTS_DIR = Path("results")

def read_csv(path):
    with path.open(newline="", encoding="utf-8") as source:
        reader = csv.DictReader(source)
        if reader.fieldnames is None:
            raise ValueError(f"{path} must have a header row.")
        return reader.fieldnames, list(reader)


if EMPLOYMENT_PATH.exists() and EXPOSURE_PATH.exists():
    employment_columns, employment = read_csv(EMPLOYMENT_PATH)
    exposure_columns, exposure = read_csv(EXPOSURE_PATH)
else:
    employment = None
    exposure = None
    print("Add both input CSVs described in data/README.md to run the analysis.")

In [ ]:
if employment is not None and exposure is not None:
    required_employment = {"industry", "year", "employment"}
    required_exposure = {"industry", "ai_exposure"}
    if not required_employment.issubset(employment_columns):
        raise ValueError(f"Employment data must contain: {sorted(required_employment)}")
    if not required_exposure.issubset(exposure_columns):
        raise ValueError(f"Exposure data must contain: {sorted(required_exposure)}")
    if not employment or not exposure:
        raise ValueError("Input CSVs must contain at least one data row.")

    employment_by_industry = {}
    for row in employment:
        industry = row["industry"].strip()
        year = int(row["year"])
        value = float(row["employment"])
        if not industry:
            raise ValueError("Employment data contains a blank industry label.")
        records = employment_by_industry.setdefault(industry, {})
        if year in records:
            raise ValueError(f"Duplicate employment record for {industry} in {year}.")
        records[year] = value

    exposure_by_industry = {}
    for row in exposure:
        industry = row["industry"].strip()
        if not industry or industry in exposure_by_industry:
            raise ValueError("Exposure data must have one nonblank row per industry.")
        exposure_by_industry[industry] = float(row["ai_exposure"])

    summary = []
    for industry in sorted(employment_by_industry.keys() & exposure_by_industry.keys()):
        observations = employment_by_industry[industry]
        years = sorted(observations)
        first_value, last_value = observations[years[0]], observations[years[-1]]
        change_pct = (last_value / first_value - 1) * 100 if first_value else ""
        summary.append({
            "industry": industry,
            "first_year": years[0],
            "last_year": years[-1],
            "first_employment": first_value,
            "last_employment": last_value,
            "employment_change_pct": change_pct,
            "ai_exposure": exposure_by_industry[industry],
        })

    RESULTS_DIR.mkdir(exist_ok=True)
    with (RESULTS_DIR / "industry_employment_changes.csv").open("w", newline="", encoding="utf-8") as output:
        writer = csv.DictWriter(output, fieldnames=summary[0].keys() if summary else ["industry"])
        writer.writeheader()
        writer.writerows(summary)
    for row in summary:
        print(row)

    valid = [row for row in summary if row["employment_change_pct"] != ""]
    x = [row["ai_exposure"] for row in valid]
    y = [row["employment_change_pct"] for row in valid]
    if len(valid) < 2 or len(set(x)) < 2 or len(set(y)) < 2:
        correlation = math.nan
    else:
        x_mean, y_mean = sum(x) / len(x), sum(y) / len(y)
        numerator = sum((a - x_mean) * (b - y_mean) for a, b in zip(x, y))
        denominator = math.sqrt(sum((a - x_mean) ** 2 for a in x) * sum((b - y_mean) ** 2 for b in y))
        correlation = max(-1, min(1, numerator / denominator))
    print("Pearson correlation:", correlation)

## Interpretation

Review the input definitions, units, industry coverage, and time period before interpreting the descriptive correlation. The output table is saved to `results/industry_employment_changes.csv`; no result should be treated as causal evidence.